# 🌾 Smart Rice Guard — Klasifikasi Penyakit Tanaman Padi
Notebook ini melatih dua model (Decision Tree & KNN) untuk mengklasifikasikan penyakit tanaman padi berdasarkan data lingkungan.

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 1 — IMPORT LIBRARY
# ══════════════════════════════════════════════════════════════
import pickle
import warnings
from pathlib import Path

# Sembunyikan UserWarning sklearn tentang kelas minoritas pada CV
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

print("✔  Library berhasil diimport.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 2 — KONFIGURASI
# ══════════════════════════════════════════════════════════════
DEFAULT_CSV = "Dataset_Lingkungan_Penyakit_Padi_ID.csv"

FEATURES = [
    "Suhu_Maks_C",
    "Total_Curah_Hujan",
    "Kelembapan_Tanah_Akar",
    "Kelembapan_Udara_%",
]
TARGET      = "Label_Penyakit"
TEST_SIZE   = 0.2
RANDOM_SEED = 42

print("✔  Konfigurasi selesai.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 3 — FUNGSI PEMBANTU
# ══════════════════════════════════════════════════════════════
def _sep(char="─", n=60):
    print(char * n)

def _save(obj, path: str):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  ✔  Disimpan → {path}")

print("✔  Fungsi pembantu siap.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 4 — [1/5] MUAT DATASET
# ══════════════════════════════════════════════════════════════
csv_path = DEFAULT_CSV   # ← ganti path di sini jika perlu

_sep("═")
print("  Smart Rice Guard — Training Script  ")
_sep("═")
print(f"\n[1/5] Memuat dataset: {csv_path}")

if not Path(csv_path).exists():
    raise FileNotFoundError(f"File tidak ditemukan: {csv_path}")

df = pd.read_csv(csv_path)
print(f"  ✔  {len(df)} baris · {df[TARGET].nunique()} kelas")
print(f"\n  Distribusi kelas:")
for label, count in df[TARGET].value_counts().items():
    bar = "█" * count
    print(f"    {label:<30} {count:>3}  {bar}")

# ── Siapkan X dan y ─────────────────────────────────────────
X = df[FEATURES].values
y = df[TARGET].values

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 5 — [2/5] SPLIT DATA & SCALING
# ══════════════════════════════════════════════════════════════
print(f"\n[2/5] Split data (test={int(TEST_SIZE*100)}%) & StandardScaler")

# Kelas 'Penyakit Garis Daun' hanya 1 sampel → tidak bisa stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
X_all_s   = scaler.transform(X)

print(f"  ✔  Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 6 — [3/5] DEFINISI MODEL
# ══════════════════════════════════════════════════════════════
print("\n[3/5] Mendefinisikan model")

models = {
    "Decision Tree": DecisionTreeClassifier(
        criterion="gini",
        max_depth=None,        # biarkan tumbuh penuh
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_SEED,
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=1,         # K=1 terbaik pada LOO-CV untuk dataset ini
        weights="distance",
        metric="minkowski",
        p=2,                   # Euclidean
    ),
}

print(f"  ✔  {len(models)} model didefinisikan: {', '.join(models.keys())}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 7 — [4/5] TRAINING & EVALUASI
# ══════════════════════════════════════════════════════════════
print("\n[4/5] Training & Evaluasi\n")
_sep()

results = {}
for name, clf in models.items():
    print(f"\n  ▶  {name}")
    clf.fit(X_train_s, y_train)

    y_pred    = clf.predict(X_test_s)
    test_acc  = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(clf, X_all_s, y, cv=5, scoring="accuracy")

    results[name] = {
        "model":    clf,
        "test_acc": test_acc,
        "cv_mean":  cv_scores.mean(),
        "cv_std":   cv_scores.std(),
    }

    print(f"     Test Accuracy : {test_acc*100:.2f}%")
    print(f"     CV-5 Accuracy : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
    print()
    print("  Classification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    _sep()

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 8 — [5/5] SIMPAN ARTEFAK MODEL
# ══════════════════════════════════════════════════════════════
print("\n[5/5] Menyimpan artefak model\n")

# Scaler & features (bersama)
_save(scaler,   "scaler_padi.pkl")
_save(FEATURES, "model_features.pkl")

# Decision Tree
dt_acc = round(results["Decision Tree"]["test_acc"] * 100, 2)
_save(results["Decision Tree"]["model"], "model_padi_dt.pkl")
_save(dt_acc, "model_accuracy_dt.pkl")
_save(dt_acc, "model_accuracy.pkl")    # kompatibilitas GUI lama

# KNN
knn_acc = round(results["K-Nearest Neighbors"]["test_acc"] * 100, 2)
_save(results["K-Nearest Neighbors"]["model"], "model_padi_knn.pkl")
_save(knn_acc, "model_accuracy_knn.pkl")

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEL 9 — RINGKASAN AKHIR
# ══════════════════════════════════════════════════════════════
_sep("═")
print("\n  RINGKASAN PERFORMA MODEL\n")
print(f"  {'Model':<25} {'Test Acc':>10}  {'CV-5 Mean':>10}  {'CV-5 Std':>10}")
_sep()
for name, r in results.items():
    print(
        f"  {name:<25}"
        f"  {r['test_acc']*100:>8.2f}%"
        f"  {r['cv_mean']*100:>8.2f}%"
        f"  ±{r['cv_std']*100:>6.2f}%"
    )
_sep("═")
print("\n  ✅ Semua file .pkl berhasil dibuat. Jalankan GUI.py sekarang.\n")